# HoWDe Home/Work Detection — Parallel Run

Processes all `.parquet` files in `paths.input_dir` (configured in `06_howde_config.yaml`) concurrently via `ThreadPoolExecutor`.
One output file is written per input file to `paths.output_dir` with `processing.output_suffix` appended to the stem (e.g. `00.parquet` → `00_howde.parquet`).
Already-completed files are skipped so interrupted runs can be resumed.

In [2]:
import os
import glob
import time
import yaml
from concurrent.futures import ThreadPoolExecutor, as_completed

import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from howde import HoWDe_labelling

with open("06_howde_config.yaml") as fp:
    config = yaml.safe_load(fp)

cfg_paths = config["paths"]
cfg_howde = config["howde"]
cfg_spark = config["spark"]
cfg_proc  = config["processing"]

# spark.driver.memory must reach the JVM before it starts — SparkSession.config is too late
os.environ["PYSPARK_SUBMIT_ARGS"] = f"--driver-memory {cfg_spark['driver_memory']} pyspark-shell"

### Spark session

`spark.scheduler.mode=FAIR` is required so that concurrent queries submitted by the `ThreadPoolExecutor` workers truly interleave on available executor threads rather than queuing up (FIFO default would defeat the parallelism).

`shuffle.partitions` is set relative to CPU count for local runs — **remove that line for cluster runs**.

In [3]:
spark = (
    SparkSession.builder
    .appName("HoWDeParallel")
    .config("spark.scheduler.mode", cfg_spark["scheduler_mode"])
    .config("spark.sql.shuffle.partitions", str(os.cpu_count() * cfg_spark["shuffle_partitions_per_cpu"]))  # local only — remove for cluster
    .config("spark.sql.execution.arrow.pyspark.enabled", str(cfg_spark["arrow_enabled"]).lower())
    .config("spark.sql.session.timeZone", cfg_spark["timezone"])
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/21 23:02:50 WARN Utils: Your hostname, ab-vsrv09, resolves to a loopback address: 127.0.1.1; using 10.50.152.25 instead (on interface ens160)
26/05/21 23:02:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 23:02:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Per-file processing function

Each `ThreadPoolExecutor` worker calls this function with one input path.

**Timezone offset:** the parquet stores `interval_start` as UTC epoch microseconds with a `Europe/Budapest` label. HoWDe expects `tz_hour_start` / `tz_minute_start` offsets to reconstruct local time internally. Computed by comparing UTC and local calendar strings — `from_utc_timestamp` handles CET/CEST DST automatically.

**Repartitioning:** each parquet file holds complete data per user (no user spans files). `repartitionByRange` co-locates all stops for a user on the same executor before HoWDe's window operations run.

**Output:** `.toPandas().to_parquet()` writes a single flat file with `output_suffix` appended to the stem.

In [4]:
def process_file(parquet_path):
    stem     = os.path.splitext(os.path.basename(parquet_path))[0]
    out_name = stem + cfg_proc["output_suffix"] + ".parquet"
    out_path = os.path.join(cfg_paths["output_dir"], out_name)

    if os.path.exists(out_path):
        return {"file": out_name, "status": "skipped"}

    t0 = time.perf_counter()
    try:
        raw = spark.read.parquet(parquet_path)

        # UTC offset per stop (accounts for DST: CET = +1h, CEST = +2h)
        utc_str     = F.date_format(F.col("interval_start"), "yyyy-MM-dd HH:mm:ss")
        local_str   = F.date_format(
            F.from_utc_timestamp(F.col("interval_start"), cfg_proc["local_timezone"]),
            "yyyy-MM-dd HH:mm:ss"
        )
        offset_secs = F.unix_timestamp(local_str) - F.unix_timestamp(utc_str)

        input_data = (
            raw
            .withColumnRenamed("device_id", "useruuid")
            .withColumnRenamed("stop_id", "loc")
            .withColumn("start", F.unix_timestamp(F.col("interval_start")))
            .withColumn("end",   F.unix_timestamp(F.col("interval_stop")))
            .withColumn("tz_hour_start",   (offset_secs / 3600).cast("long"))
            .withColumn("tz_minute_start", ((offset_secs % 3600) / 60).cast("long"))
            .withColumn("country", F.lit(cfg_proc["country"]))
            .select("useruuid", "loc", "start", "end", "tz_hour_start", "tz_minute_start", "country")
        )
        input_data = input_data.repartitionByRange(os.cpu_count(), "useruuid")

        labeled = HoWDe_labelling(input_data, **cfg_howde, verbose=False)
        pdf = labeled.toPandas()
        pdf.to_parquet(out_path, index=False)

        return {"file": out_name, "status": "ok", "rows": len(pdf), "elapsed": time.perf_counter() - t0}

    except Exception as exc:
        return {"file": out_name, "status": "error", "error": str(exc), "elapsed": time.perf_counter() - t0}

### Discover input files and prepare output directory

In [7]:
input_files = sorted(glob.glob(os.path.join(cfg_paths["input_dir"], "*.parquet")))
#input_files = sorted(glob.glob(os.path.join(cfg_paths["input_dir"], "*.parquet")))[:10]
os.makedirs(cfg_paths["output_dir"], exist_ok=True)

print(f"Input files found : {len(input_files)}")
print(f"Output directory  : {cfg_paths['output_dir']}")
print(f"Max parallel files: {cfg_proc['max_parallel_files']}")

Input files found : 256
Output directory  : /mnt/common-ssd/group-dirs/tkom-2026-project/pickwell/budapest/howde/v1
Max parallel files: 4


### Run parallel HoWDe

Files already present in `output_dir` are skipped automatically — safe to re-run after interruption.

In [8]:
t_total = time.perf_counter()
results = []

with ThreadPoolExecutor(max_workers=cfg_proc["max_parallel_files"]) as executor:
    futures = {executor.submit(process_file, p): p for p in input_files}
    for i, future in enumerate(as_completed(futures), 1):
        res = future.result()
        results.append(res)
        if res["status"] == "ok":
            print(f"[{i:>3}/{len(input_files)}] OK      {res['file']}  {res['rows']:>8} rows  {res['elapsed']:.1f}s")
        elif res["status"] == "skipped":
            print(f"[{i:>3}/{len(input_files)}] SKIPPED {res['file']}")
        else:
            print(f"[{i:>3}/{len(input_files)}] ERROR   {res['file']}  {res['error']}")

print(f"\nTotal elapsed: {time.perf_counter() - t_total:.1f}s")

[  1/256] SKIPPED 04_howde.parquet
[  2/256] SKIPPED 02_howde.parquet
[  3/256] SKIPPED 01_howde.parquet
[  4/256] SKIPPED 00_howde.parquet
[  5/256] SKIPPED 06_howde.parquet
[  6/256] SKIPPED 07_howde.parquet
[  7/256] SKIPPED 08_howde.parquet
[  8/256] SKIPPED 0a_howde.parquet
[  9/256] SKIPPED 0b_howde.parquet
[ 10/256] SKIPPED 0c_howde.parquet
[ 11/256] SKIPPED 05_howde.parquet
[ 12/256] SKIPPED 0e_howde.parquet
[ 13/256] SKIPPED 0f_howde.parquet
[ 14/256] SKIPPED 10_howde.parquet
[ 15/256] SKIPPED 11_howde.parquet
[ 16/256] SKIPPED 0d_howde.parquet
[ 17/256] SKIPPED 03_howde.parquet
[ 18/256] SKIPPED 14_howde.parquet
[ 19/256] SKIPPED 15_howde.parquet
[ 20/256] SKIPPED 13_howde.parquet
[ 21/256] SKIPPED 17_howde.parquet
[ 22/256] SKIPPED 18_howde.parquet
[ 23/256] SKIPPED 12_howde.parquet
[ 24/256] SKIPPED 1a_howde.parquet
[ 25/256] SKIPPED 1b_howde.parquet
[ 26/256] SKIPPED 1c_howde.parquet
[ 27/256] SKIPPED 1d_howde.parquet
[ 28/256] SKIPPED 1e_howde.parquet
[ 29/256] SKIPPED 1f

26/05/21 23:03:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/21 23:03:24 WARN BlockManager: Block rdd_50_3 already exists on this machine; not re-adding it
26/05/21 23:03:25 WARN BlockManager: Block rdd_48_18 already exists on this machine; not re-adding it
26/05/21 23:03:25 WARN BlockManager: Block rdd_50_12 already exists on this machine; not re-adding it
26/05/21 23:03:25 WARN BlockManager: Block rdd_50_11 already exists on this machine; not re-adding it
26/05/21 23:03:25 WARN BlockManager: Block rdd_50_18 already exists on this machine; not re-adding it


[ 46/256] OK      2c_howde.parquet    258358 rows  50.1s


[ 47/256] OK      2f_howde.parquet    265811 rows  68.4s


[ 48/256] OK      2e_howde.parquet    275374 rows  71.2s


...

[252/256] OK      fb_howde.parquet    384922 rows  42.8s


[253/256] OK      fc_howde.parquet    257083 rows  42.2s


[254/256] OK      fd_howde.parquet    256199 rows  43.0s


[255/256] OK      fe_howde.parquet    270851 rows  36.9s


[256/256] OK      ff_howde.parquet    275536 rows  30.8s

Total elapsed: 2176.9s


### Summary

In [7]:
ok      = [r for r in results if r["status"] == "ok"]
skipped = [r for r in results if r["status"] == "skipped"]
errors  = [r for r in results if r["status"] == "error"]

print(f"Completed : {len(ok)}")
print(f"Skipped   : {len(skipped)}")
print(f"Errors    : {len(errors)}")
if ok:
    print(f"Total rows: {sum(r['rows'] for r in ok):,}")
if errors:
    print("\nFailed files:")
    for r in errors:
        print(f"  {r['file']}: {r['error']}")

Completed : 10
Skipped   : 0
Errors    : 0
Total rows: 3,123,742


---
## Results inspection

Load one output file and print basic stats: label distribution, unique users with home/work.
Change `inspect_file` to point at any other `*_howde.parquet` you want to examine.

In [ ]:
import polars as pl

# --- pick the file to inspect -------------------------------------------
inspect_file = sorted(glob.glob(os.path.join(cfg_paths['output_dir'], '*_howde.parquet')))[0]
# -------------------------------------------------------------------------

df = pl.read_parquet(inspect_file)
print(f"File      : {os.path.basename(inspect_file)}")
print(f"Rows      : {len(df):,}")
print(f"Columns   : {df.columns}")
df.head(3)

In [ ]:
label_col = "place_label"   # adjust if HoWDe uses a different column name

# --- label distribution --------------------------------------------------
print("=== Label distribution ===")
print(df[label_col].value_counts(sort=True))

# --- users with at least one home / work ---------------------------------
print("\n=== Users with home / work ===")
for lbl in ["H", "W"]:   # HoWDe uses 'H' and 'W' by default
    n = df.filter(pl.col(label_col) == lbl)["useruuid"].n_unique()
    total = df["useruuid"].n_unique()
    print(f"  {lbl}: {n:,} / {total:,} users  ({100*n/total:.1f}%)")

# --- stops per label per user (median / mean) ----------------------------
print("\n=== Stops per user (median | mean) ===")
(
    df.group_by(["useruuid", label_col])
      .agg(pl.len().alias("n_stops"))
      .group_by(label_col)
      .agg(
          pl.col("n_stops").median().alias("median"),
          pl.col("n_stops").mean().alias("mean"),
      )
      .sort(label_col)
      .with_columns([
          pl.col("median").round(1),
          pl.col("mean").round(1),
      ])
)